In [72]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [73]:
import torch
from datasets import load_dataset
from torchvision.transforms import v2
import dotenv
import os

In [74]:
dotenv.load_dotenv()
#print(os.getenv("HF_TOKEN")) #HF_TOKEN prescence in envvars is automatically detected by huggingface functions

True

In [75]:
hf_dataset_training = load_dataset("cifar10", split="train[:5%]")
hf_dataset_val_test = load_dataset("cifar10", split="test[:2%]")

train_transform = v2.Compose([
    v2.ToImage(), # converts into Image
    v2.RandomCrop(32, padding=4), # randomly crops a portion of the image 
    v2.RandomHorizontalFlip(p=0.5), #mirror 50% of the images
    v2.ToDtype(torch.float32, scale=True),# converts to tensor with datatype float32 
    v2.Normalize(mean=[0.4914, 0.4822, 0.4465], std=[0.2023, 0.1994, 0.2010]) #z-score normalization for the 3-channels
])

class CIFAR10(torch.utils.data.Dataset):
    def __init__(self, hf_dataset, transform):
        self.dataset = hf_dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx] #{label, image}
        image = self.transform(item['img']) #apply transform to images independently
        label = item['label']
        return image, label

dataset_train = CIFAR10(hf_dataset_training, train_transform)
print(len(dataset_train))
dataset_val, dataset_test = hf_dataset_val_test[:len(hf_dataset_val_test)//2], hf_dataset_val_test[len(hf_dataset_val_test)//2:]

2500


In [76]:
train_loader = torch.utils.data.DataLoader(dataset_train, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(dataset_test, batch_size=64, shuffle=True)
val_loader = torch.utils.data.DataLoader(dataset_val, batch_size=64, shuffle=True)

In [77]:
import architectures.BasicNet

In [78]:
BasicNet = architectures.BasicNet.BasicNet()

In [ ]:
import torch
import torch.nn as nn

def train(model, dataloader_train,dataload_val):
    model.train()
    count = 0
    optimizer = torch.optim.SGD(model.parameters(), lr=1e-3) #minibatch
    for batch_index, (image, label) in enumerate(dataloader_train):
        optimizer.zero_grad() #zero out the gradients

        #print(image.shape)

        #output = model(image.reshape(image.size(0), -1))

        sample = torch.randint(0,1,size=(64,9999),dtype=torch.float32)
        output=model(sample)
        print(output.shape)
        #loss = torch.nn.functional.cross_entropy(output, label)
        #loss.backward()# computer gradients
        #ptimizer.step()

class BasicNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers=nn.Linear(9999,2)
        #self.layers=nn.Sequential(nn.Conv2d(in_channels=3, out_channels=96, kernel_size=(11,11), stride=4),
        #    nn.ReLU(), 
        #    nn.MaxPool2d((3,3), stride=2),
        #    nn.Linear(4096,1000),
        #    nn.Linear(1000,10))
        
    def forward(self,x):
        return self.layers(x)        
    


In [94]:
bn = BasicNet()
train(bn,train_loader, val_loader)

RuntimeError: mat1 and mat2 must have the same dtype, but got Long and Float